In [1]:
!pip install -q -U langgraph langchain langchain-groq


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [2]:
# Install required packages
%pip install -U langgraph langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import required modules
import os
from getpass import getpass
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [4]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


In [5]:
# Import ChatGroq
from langchain_groq import ChatGroq

# Create Groq LLM using GPT-OSS
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Send prompt
response = llm.invoke("Explain LangGraph in one sentence")

# Print response
print(response.content)

LangGraph is a framework that lets developers design, orchestrate, and execute complex language‑model workflows as directed graphs, enabling modular, stateful, and context‑aware AI applications.


### FIRST STATE GRAPH

In [6]:
# Generate an AI topic using the LLM
topic = llm.invoke("Generate an AI topic").content

# Generate a 100-word article on the generated topic
article = llm.invoke(f"Write 100 words about {topic}").content

# Print the generated article
print(article)

Federated learning (FL) offers a promising framework for privacy‑preserving medical imaging, enabling hospitals to collaboratively train deep‑learning models without exchanging raw patient scans. By exchanging only model updates, FL reduces data‑sharing risks while leveraging diverse datasets. Research focuses on communication‑efficient protocols, such as sparsification and quantization, to mitigate bandwidth constraints. Differential privacy mechanisms further protect individual contributions, ensuring that updates reveal no sensitive information. Addressing data heterogeneity—different scanner types, protocols, and patient demographics—requires robust aggregation strategies and domain adaptation techniques. Ultimately, FL can enhance diagnostic accuracy across institutions while upholding stringent confidentiality standards. This approach fosters trust and accelerates innovation.


In [7]:
# Define the structure of the graph state
class State(TypedDict):
    topic: str

In [8]:
# Define a node that generates an AI topic
def topic_generator(state):
    # Ask the LLM to generate an AI topic
    topic = llm.invoke("Generate an AI topic").content
    # Return the generated topic to update the state
    return {"topic": topic}

In [9]:
# Create a StateGraph using our State structure
builder = StateGraph(State)

# Add the topic generator node to the graph
builder.add_node("topic_generator", topic_generator)

# Connect START to the topic generator
builder.add_edge(START, "topic_generator")

# Connect the topic generator to END
builder.add_edge("topic_generator", END)

graph=builder.compile()

In [10]:
result=graph.invoke({})

In [11]:
print(result)

{'topic': '**AI Topic:**  \n**Federated Learning for Privacy‑Preserving Medical Imaging**\n\n**Brief Description:**  \nInvestigate how federated learning (FL) can enable hospitals to collaboratively train deep‑learning models on sensitive medical imaging data (e.g., MRIs, CT scans) without sharing raw patient images. The research would explore communication‑efficient FL algorithms, differential privacy guarantees, and robustness to heterogeneous data distributions across institutions. The goal is to improve diagnostic accuracy while strictly protecting patient confidentiality.'}


### STATE MANAGEMENT

In [12]:
# Define the topic
topic = "Machine Learning"

# Generate an article about the topic
article = llm.invoke(f"Write about {topic}").content

# Summarize the generated article
summary = llm.invoke(f"Summarize this\n{article}").content

# Print the final summary
print(summary)

**Machine Learning – Quick Overview**

| Section | Key Take‑aways |
|---------|----------------|
| **What ML Is** | AI subfield that learns patterns from data, improving over time without explicit programming. |
| **Core Idea** | Train a model on input–output pairs (training data) to predict unseen outputs; learning can be supervised, unsupervised, semi‑supervised, or reinforcement. |
| **Historical Milestones** | 1950s‑Perceptron → 1967‑Backprop → 1980s‑SVM → 1995‑Random Forest → 2006‑Deep Learning revival → 2012‑AlexNet → 2018‑Transformer → 2020s‑Foundation models (GPT‑4, LLaMA, Stable Diffusion). |
| **ML Types & Use Cases** | • **Supervised** – spam, image classification, diagnosis. <br>• **Unsupervised** – clustering, anomaly detection. <br>• **Semi‑Supervised** – speech, web mining. <br>• **Reinforcement** – robotics, games, autonomous driving. <br>• **Self‑Supervised** – pre‑training language/vision models. <br>• **Federated** – privacy‑preserving mobile/medical training. |
| **

In [13]:
# Define the structure of the graph state
class State(TypedDict):
    topic: str
    article: str
    summary: str

# Define the writer node
def writer(state):
    # Generate an article using the topic from the state
    article = llm.invoke(f"Write about {state['topic']}").content
    # Update the article in the state
    return {"article": article}

# Define the summarizer node
def summarizer(state):
    # Summarize the article from the state
    summary = llm.invoke(f"Summarize this\n{state['article']}").content
    # Update the summary in the state
    return {"summary": summary}

# Create the StateGraph
builder = StateGraph(State)

# Add nodes to the graph
builder.add_node("writer", writer)
builder.add_node("summarizer", summarizer)

# Define the flow of the graph
builder.add_edge(START, "writer")
builder.add_edge("writer", "summarizer")
builder.add_edge("summarizer", END)

# Compile the graph
graph = builder.compile()

In [14]:
# Run the graph with "Machine Learning" as the input topic
result = graph.invoke({"topic": "Machine Learning"})

# Print the final summary generated by the summarizer node
print(result["summary"])

**Machine Learning – Quick Overview**

| Section | Key Take‑aways |
|---------|----------------|
| **What is ML?** | AI subfield that builds models which learn patterns from data and generalize to new cases, reducing the need for hand‑coded rules. |
| **Core Idea** | A learning algorithm maps input–output pairs (training data) to a function that predicts or classifies unseen data; quality is measured by predictive accuracy. |
| **Historical Milestones** | 1950s‑Perceptron → 1960s‑Decision Trees → 1980s‑Backpropagation → 1990s‑SVM → 2000s‑Random Forests → 2010s‑Deep CNNs (AlexNet) → 2020s‑Transformers (BERT, GPT). |
| **ML Types** | • Supervised (labels) – e.g., SVM, neural nets<br>• Unsupervised – e.g., k‑means, PCA<br>• Semi‑supervised – mix of labeled/unlabeled<br>• Reinforcement – learns via rewards (Q‑learning, policy gradients)<br>• Self‑supervised – generates its own labels (contrastive learning)<br>• Federated – trains across devices while keeping data local |
| **Core Concepts 

### CONDITIONAL ROUTING

In [15]:
# Example question to test the routing logic
question = "Write Python code"

# Check whether the question contains the word "code"
if "code" in question.lower():
    print("Coding Agent")
else:
    print("General Agent")

# Define the structure of the graph state
class State(TypedDict):
    question: str
    route: str
    answer: str

# Define the router node
def router(state):
    # By default, route the question to the coding agent
    route = "coding"

    # If "code" is not present, route to the general agent
    if "code" not in state["question"].lower():
        route = "general"

    # Store the selected route in the state
    return {"route": route}

# Define the coding agent node
def coding_agent(state):
    # Return response from the coding agent
    return {"answer": "Coding Agent Selected"}

# Define the general agent node
def general_agent(state):
    # Return response from the general agent
    return {"answer": "General Agent Selected"}

Coding Agent


In [16]:
# Define the decision function
# It returns the route selected by the router node
def decide(state):
    return state["route"]

# Create the StateGraph
builder = StateGraph(State)

# Add all nodes to the graph
builder.add_node("router", router)
builder.add_node("coding", coding_agent)
builder.add_node("general", general_agent)

# Start the graph with the router node
builder.add_edge(START, "router")

# Route conditionally based on the value returned by decide()
builder.add_conditional_edges(
    "router",
    decide,
    {
        "coding": "coding",
        "general": "general"
    }
)

# End the graph after either agent finishes
builder.add_edge("coding", END)
builder.add_edge("general", END)

# Compile the graph
graph = builder.compile()

In [17]:
# Run the graph with a coding-related question
result = graph.invoke({"question": "Write Python code"})

# Print the answer returned by the selected agent
print(result["answer"])

Coding Agent Selected


### MESSAGE STATE

In [18]:
# First question tells the LLM your name
question1 = "My name is AB"

# Send the first question to the LLM
llm.invoke(question1)

# Ask the LLM to remember your name
question2 = "What is my name"

# Get the response from the LLM
answer = llm.invoke(question2).content

# Print the answer
print(answer)

I’m not sure what your name is—could you let me know?


In [19]:
# Import MessagesState for storing conversation messages
from langgraph.graph import MessagesState

# Import HumanMessage for creating user messages
from langchain_core.messages import HumanMessage

In [20]:
# Define the chatbot node
def chatbot(state):
    # Send all messages stored in the state to the LLM
    response = llm.invoke(state["messages"])

    # Add the LLM response to the message history
    return {"messages": [response]}

In [21]:
# Create a StateGraph using MessagesState
builder = StateGraph(MessagesState)

# Add the chatbot node
builder.add_node("chatbot", chatbot)

# Define the graph flow
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# Compile the graph
graph = builder.compile()

In [22]:
# Send the first message to the graph
result = graph.invoke({
    "messages": [
        HumanMessage(content="My name is AB")
    ]
})

# Print the latest response from the chatbot
print(result["messages"][-1].content)

Nice to meet you, AB! How can I help you today?


In [23]:
# Send both messages together so the LLM has the conversation context
result = graph.invoke({
    "messages": [
        HumanMessage(content="My name is AB"),
        HumanMessage(content="What is my name")
    ]
})

# Print the latest response
print(result["messages"][-1].content)

You’re AB.


In [24]:
# Send only this message in a new graph invocation
# The graph will NOT remember the previous invocation yet
result = graph.invoke({
    "messages": [
        HumanMessage(content="What is my name")
    ]
})

# Print the latest response
print(result["messages"][-1].content)

I’m not sure what your name is—could you let me know?


### BUILDING A CHATBOT

In [25]:
# Take a question as input from the user
question = input("You: ")

# Send the question to the LLM
response = llm.invoke(question).content

# Print the LLM response
print(response)

I’m sorry, but I can’t help with that.


In [26]:
# Define the chatbot node
def chatbot(state):
    # Send the conversation messages to the LLM
    response = llm.invoke(state["messages"])

    # Add the LLM response to the message history
    return {"messages": [response]}

In [27]:
# Create a StateGraph using MessagesState
builder = StateGraph(MessagesState)

# Add the chatbot node
builder.add_node("chatbot", chatbot)

# Connect START to chatbot
builder.add_edge(START, "chatbot")

# Connect chatbot to END
builder.add_edge("chatbot", END)

# Compile the graph
graph = builder.compile()

In [28]:
# Import HumanMessage to create user messages
from langchain_core.messages import HumanMessage

In [29]:
# Create an empty list to store the complete conversation
conversation = []

# Start an infinite chat loop
while True:
    # Take input from the user
    user_input = input("You: ")

    # Stop the chatbot if the user types "exit"
    if user_input.lower() == "exit":
        break

    # Add the user's message to the conversation history
    conversation.append(HumanMessage(content=user_input))

    # Send the complete conversation history to the graph
    result = graph.invoke({"messages": conversation})

    # Update conversation with all messages including the AI response
    conversation = result["messages"]

    # Print the latest AI response
    print("AI:", conversation[-1].content)

AI: Got it—request cancelled. If you need anything else, just let me know!
AI: Request cancelled. If you need anything else, just let me know!
AI: Request cancelled.
AI: Request cancelled.
AI: Request cancelled.
AI: 
AI: Request cancelled.


### Tool Calling with Nodes

In [30]:
# Ask the LLM about the current weather
# Note: LLM alone does not have access to live weather data
response = llm.invoke("What is the weather in Mumbai right now").content

# Print the response
print(response)

I don’t have real‑time access to weather data, so I can’t tell you the exact temperature, humidity, or forecast for Mumbai at this moment.  

**What you can do instead:**

| Option | How to use it | Typical info you’ll get |
|--------|---------------|------------------------|
| **Weather websites** (e.g., Weather.com, AccuWeather, BBC Weather) | Go to the site, type “Mumbai” in the search bar. | Current temperature, wind speed, humidity, chance of rain, 7‑day forecast. |
| **Weather apps** (e.g., The Weather Channel app, Weather Underground, Google Weather) | Open the app, add Mumbai as a location. | Same as above, plus alerts and radar. |
| **Voice assistants** (Google Assistant, Siri, Alexa) | Say “Hey Google, what’s the weather in Mumbai?” | Quick spoken summary. |
| **APIs** (OpenWeatherMap, WeatherAPI, AccuWeather API) | Use an API key and call the endpoint for current weather. | JSON with temperature, pressure, wind, etc. |

If you need the data for a script or a project, you can

In [31]:
# Import requests to call the free weather API
import requests

# Define the structure of the graph state
class State(TypedDict):
    city: str
    weather: str

In [32]:
# Define the weather node
def weather_node(state):
    # Free Open-Meteo geocoding API
    url = "https://geocoding-api.open-meteo.com/v1/search"

    # Find latitude and longitude of the city
    geo = requests.get(
        url,
        params={
            "name": state["city"],
            "count": 1
        }
    ).json()

    # Extract latitude and longitude
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # Get current weather using the free Open-Meteo API
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m"
        }
    ).json()

    # Extract current temperature
    temp = weather["current"]["temperature_2m"]

    # Store the temperature in the state
    return {"weather": f"{temp}°C"}

In [33]:
# Create the StateGraph
builder = StateGraph(State)

# Add the weather node to the graph
builder.add_node("weather", weather_node)

# Connect START to the weather node
builder.add_edge(START, "weather")

# Connect the weather node to END
builder.add_edge("weather", END)

# Compile the graph
graph = builder.compile()

In [34]:
# Run the graph with Mumbai as the city
result = graph.invoke({"city": "Mumbai"})

# Print the current temperature
print(result["weather"])

26.6°C


In [35]:
# Define the structure of the graph state
class State(TypedDict):
    city: str
    weather: str
    answer: str

In [36]:
# Define the weather node
def weather_node(state):
    # Free Open-Meteo geocoding API
    url = "https://geocoding-api.open-meteo.com/v1/search"

    # Find latitude and longitude of the city
    geo = requests.get(
        url,
        params={
            "name": state["city"],
            "count": 1
        }
    ).json()

    # Extract latitude and longitude
    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    # Get current weather using the free Open-Meteo API
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m"
        }
    ).json()

    # Extract current temperature
    temp = weather["current"]["temperature_2m"]

    # Store the temperature in the state
    return {"weather": f"{temp}°C"}

In [37]:
# Define the formatter node
def formatter(state):
    # Convert the raw weather data into a user-friendly response
    answer = llm.invoke(
        f"Convert this weather data into a user friendly response: {state['weather']}"
    ).content

    # Store the formatted response in the state
    return {"answer": answer}

In [38]:
# Create the StateGraph
builder = StateGraph(State)

# Add the weather node to get weather data
builder.add_node("weather", weather_node)

# Add the formatter node to create a user-friendly response
builder.add_node("formatter", formatter)

# Start the graph with the weather node
builder.add_edge(START, "weather")

# Send weather data to the formatter node
builder.add_edge("weather", "formatter")

# End the graph after formatting the response
builder.add_edge("formatter", END)

# Compile the graph
graph = builder.compile()

### Parallel Execution

In [39]:
# Run the three LLM calls normally one after another
ai = llm.invoke("Explain AI").content
ml = llm.invoke("Explain Machine Learning").content
dl = llm.invoke("Explain Deep Learning").content

# Print all the responses
print(ai)
print(ml)
print(dl)

## What Is Artificial Intelligence (AI)?

Artificial Intelligence (AI) is the field of computer science that builds systems capable of performing tasks that normally require human intelligence. These tasks include:

| Task | Human‑like AI Capability | Typical AI Technique |
|------|--------------------------|----------------------|
| Recognizing faces in photos | **Computer vision** | Convolutional Neural Networks (CNNs) |
| Understanding spoken language | **Speech recognition** | Recurrent Neural Networks (RNNs), Transformers |
| Translating between languages | **Natural language processing** | Sequence‑to‑sequence models, BERT, GPT |
| Playing chess or Go | **Strategic reasoning** | Reinforcement learning, Monte‑Carlo tree search |
| Driving a car | **Perception + decision‑making** | Sensor fusion, deep reinforcement learning |
| Diagnosing diseases | **Pattern recognition** | Deep learning on medical imaging, rule‑based systems |

In short, AI systems **observe**, **learn**, and **a

In [40]:
# Import TypedDict to define the graph state structure
from typing import TypedDict

# Define the structure of the graph state
class State(TypedDict):
    ai: str
    ml: str
    dl: str

In [41]:
# Define the AI node
def ai_node(state):
    # Generate an explanation of AI
    return {"ai": llm.invoke("Explain AI").content}

# Define the Machine Learning node
def ml_node(state):
    # Generate an explanation of Machine Learning
    return {"ml": llm.invoke("Explain Machine Learning").content}

# Define the Deep Learning node
def dl_node(state):
    # Generate an explanation of Deep Learning
    return {"dl": llm.invoke("Explain Deep Learning").content}

In [42]:
# Create the StateGraph
builder = StateGraph(State)

# Add all three nodes to the graph
builder.add_node("ai", ai_node)
builder.add_node("ml", ml_node)
builder.add_node("dl", dl_node)

# Connect START to all three nodes
# This allows all three nodes to execute in parallel
builder.add_edge(START, "ai")
builder.add_edge(START, "ml")
builder.add_edge(START, "dl")

# Connect all three nodes to END
builder.add_edge("ai", END)
builder.add_edge("ml", END)
builder.add_edge("dl", END)

# Compile the graph
graph = builder.compile()

In [43]:
# Run the graph
result = graph.invoke({})

# Print the AI explanation
print(result["ai"])

# Print the Machine Learning explanation
print(result["ml"])

# Print the Deep Learning explanation
print(result["dl"])

## What Is Artificial Intelligence (AI)?

Artificial Intelligence (AI) is the field of computer science that builds systems capable of performing tasks that normally require human intelligence. These tasks include:

| Task | Human‑like AI Capability | Typical AI Technique |
|------|--------------------------|----------------------|
| Recognizing faces in photos | **Computer vision** | Convolutional Neural Networks (CNNs) |
| Translating languages | **Natural language processing** | Transformer models (e.g., GPT, BERT) |
| Playing chess or Go | **Strategic reasoning** | Reinforcement learning + search algorithms |
| Driving a car | **Perception + decision‑making** | Sensor fusion + deep RL |
| Diagnosing disease | **Pattern recognition** | Supervised learning + decision trees |

In short, AI systems learn from data or rules, generalize to new situations, and act autonomously or semi‑autonomously.

---

## A Quick History

| Era | Milestone | Key Idea |
|-----|-----------|----------|
| 1

### FAN IN FAN OUT

In [44]:
# Print individual research results
print("AI Research")
print("ML Research")
print("DL Research")

# Multiple research results need consolidation

AI Research
ML Research
DL Research


In [45]:
# Define the structure of the graph state
class State(TypedDict):
    ai: str
    ml: str
    dl: str
    final_report: str

In [46]:
# Define the aggregator node
def aggregator(state):
    # Combine AI, ML, and DL research into one final report
    report = llm.invoke(
        f"""Combine these reports
AI: {state['ai']}
ML: {state['ml']}
DL: {state['dl']}"""
    ).content

    # Store the combined report in the state
    return {"final_report": report}

In [47]:
# Create the StateGraph
builder = StateGraph(State)

# Add all nodes to the graph
builder.add_node("ai", ai_node)
builder.add_node("ml", ml_node)
builder.add_node("dl", dl_node)
builder.add_node("aggregator", aggregator)

# Fan-Out: Run AI, ML, and DL nodes in parallel
builder.add_edge(START, "ai")
builder.add_edge(START, "ml")
builder.add_edge(START, "dl")

# Fan-In: Send all three results to the aggregator
builder.add_edge("ai", "aggregator")
builder.add_edge("ml", "aggregator")
builder.add_edge("dl", "aggregator")

# End the graph after aggregation
builder.add_edge("aggregator", END)

# Compile the graph
graph = builder.compile()

In [50]:
result=graph.invoke({})
print(result["final_report"])

# AI, Machine Learning & Deep Learning – A Unified Overview  

> **Goal:**  Bring together the key ideas, history, techniques, applications, challenges, and future directions of Artificial Intelligence (AI), Machine Learning (ML), and Deep Learning (DL) into one coherent, easy‑to‑read document.  

> **Audience:**  Students, practitioners, managers, or anyone curious about how these fields fit together and why they matter today.

---

## 1.  What Is Artificial Intelligence?

Artificial Intelligence is the science of building **systems that can observe, learn, and act** in ways that mimic or surpass human cognition.  
Typical tasks include:

| Task | Human‑like AI Capability | Typical AI Technique |
|------|--------------------------|----------------------|
| Face recognition | Computer vision | Convolutional Neural Networks (CNNs) |
| Speech understanding | Speech recognition | RNNs, Transformers |
| Language translation | NLP | Seq‑to‑seq, BERT, GPT |
| Strategic games | Strategic reas

### Multi Agent Routing

In [51]:
# Ask a sample question to the LLM
question = "Write Python code for Binary Search"

# Get the direct LLM response
answer = llm.invoke(question).content

# Print the response
print(answer)

Below is a compact, well‑commented implementation of **binary search** in Python.  
It contains both an **iterative** and a **recursive** version, plus a tiny demo that shows how to use them.

```python
# ------------------------------------------------------------
# Binary Search – iterative & recursive
# ------------------------------------------------------------
def binary_search_iterative(arr, target):
    """
    Search for `target` in a sorted list `arr` using an iterative
    binary search algorithm.

    Parameters
    ----------
    arr : list[int | float]
        Sorted (ascending) list of comparable elements.
    target : int | float
        Value to locate in `arr`.

    Returns
    -------
    int
        Index of `target` in `arr` if found, otherwise -1.
    """
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        mid_val = arr[mid]

        if mid_val == target:
            return mid          # found
        elif mid_val

In [52]:
# Define the structure of the graph state
class State(TypedDict):
    question: str
    route: str
    answer: str

In [53]:
# Define the router node
def router(state):
    # Convert the question to lowercase for easier keyword checking
    q = state["question"].lower()

    # Route Python-related questions to the coding agent
    if "python" in q:
        return {"route": "coding"}

    # Route blog-related questions to the writer agent
    if "blog" in q:
        return {"route": "writer"}

    # Send everything else to the general agent
    return {"route": "general"}

In [54]:
# Define the coding agent
def coding_agent(state):
    # Ask the LLM to behave like a Python expert
    answer = llm.invoke(
        f"You are a Python expert\n{state['question']}"
    ).content

    # Store the answer in the state
    return {"answer": answer}

In [55]:
# Define the writer agent
def writer_agent(state):
    # Ask the LLM to behave like a content writer
    answer = llm.invoke(
        f"You are a content writer\n{state['question']}"
    ).content

    # Store the answer in the state
    return {"answer": answer}

In [56]:
# Define the general agent
def general_agent(state):
    # Send the question directly to the LLM
    answer = llm.invoke(state["question"]).content

    # Store the answer in the state
    return {"answer": answer}

# Define the decision function
def decide(state):
    # Return the route selected by the router
    return state["route"]

In [57]:
# Create the StateGraph
builder = StateGraph(State)

# Add all agent nodes
builder.add_node("router", router)
builder.add_node("coding", coding_agent)
builder.add_node("writer", writer_agent)
builder.add_node("general", general_agent)

# Start the graph with the router
builder.add_edge(START, "router")

# Route the question based on the value returned by decide()
builder.add_conditional_edges(
    "router",
    decide,
    {
        "coding": "coding",
        "writer": "writer",
        "general": "general"
    }
)

# End the graph after the selected agent finishes
builder.add_edge("coding", END)
builder.add_edge("writer", END)
builder.add_edge("general", END)

# Compile the graph
graph = builder.compile()

# Run the graph with a Python-related question
result = graph.invoke({
    "question": "Write Python code for Binary Search"
})

# Print the final answer
print(result["answer"])

Below is a compact, well‑documented implementation of binary search in Python.  
It includes both an **iterative** and a **recursive** version, and a small test harness so you can see it in action.

```python
#!/usr/bin/env python3
"""
Binary Search – iterative & recursive implementations

The functions assume that *arr* is sorted in ascending order.
If the target value is found, the index of the element is returned.
If the target is not present, -1 is returned.

Author: ChatGPT
"""

from typing import List, Optional


def binary_search_iterative(arr: List[int], target: int) -> int:
    """
    Perform binary search on a sorted list using an iterative approach.

    Parameters
    ----------
    arr : List[int]
        Sorted list of integers.
    target : int
        Value to search for.

    Returns
    -------
    int
        Index of target in arr, or -1 if not found.
    """
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        mid_v